In [1]:
import numpy as np
import pandas as pd

from approximate import charge_balance
from approximate.feature_engineering import MaterialFeatureExtractor

In [2]:
def select_feature_cols(df, feat_cols):
    """Keep numeric feature columns only; report anything skipped."""
    numeric = df[feat_cols].select_dtypes(include="number").columns.tolist()
    skipped = [c for c in feat_cols if c not in numeric]
    if skipped:
        print("Non-numeric features skipped:", skipped)
    return numeric

def find_near_constant(X, max_dominant=0.999, decimals=9):
    """Columns where one value covers more than `max_dominant` of rows.

    Returns a Series {col: dominant_fraction}, sorted descending.
    Rounding to `decimals` stops floating-point noise from counting as variation.
    """
    dominant = X.apply(
        lambda col: col.round(decimals).value_counts(normalize=True).iloc[0]
    )
    return dominant[dominant > max_dominant].sort_values(ascending=False)

def find_duplicates(X):
    """Exact duplicate columns. Returns {dropped_col: kept_col}."""
    kept, duplicates = [], {}
    for col in X.columns:
        match = next((k for k in kept if X[col].equals(X[k])), None)
        if match is None:
            kept.append(col)
        else:
            duplicates[col] = match
    return duplicates

def find_correlated(X, threshold=0.999, priority=None):
    """Greedy |Pearson r| pruning. Returns {dropped_col: (kept_col, r)}.

    Columns are visited in `priority` order first (any you list), then in
    their existing order, so the features you would rather keep win ties.
    """
    priority = [c for c in (priority or []) if c in X.columns]
    order = priority + [c for c in X.columns if c not in priority]
    corr = X[order].corr().abs()

    kept, correlated = [], {}
    for col in order:
        if kept:
            r = corr.loc[col, kept]
            if r.max() > threshold:
                correlated[col] = (r.idxmax(), round(float(r.max()), 4))
                continue
        kept.append(col)
    return correlated

def clean_features(df_feat, feat_cols, formula_col="Chemical", max_dominant=0.99, corr_threshold=0.99, priority=None):
    """Run the finders in order on unique compositions.

    Order: NaN guard -> near-constant -> duplicates -> correlated.
    Returns (drop_cols, report). Apply drop_cols to the full df_feat yourself.
    """
    feat_cols = select_feature_cols(df_feat, feat_cols)

    stats_df = df_feat
    X = stats_df.drop_duplicates(subset=formula_col)[feat_cols]

    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        raise ValueError(f"NaNs present - add a missing-value step first: {bad}")

    near_constant = find_near_constant(X, max_dominant)
    X = X.drop(columns=near_constant.index)

    duplicates = find_duplicates(X)
    X = X.drop(columns=list(duplicates))

    correlated = find_correlated(X, corr_threshold, priority)
    X = X.drop(columns=list(correlated))

    drop_cols = list(near_constant.index) + list(duplicates) + list(correlated)
    report = {
        "n_rows_stats": len(X),
        "n_start": len(feat_cols),
        "n_end": X.shape[1],
        "kept": X.columns.tolist(),
        "near_constant": near_constant,                      # col -> fraction
        "duplicates": pd.Series(duplicates, dtype=object),   # dropped -> kept
        "correlated": pd.DataFrame.from_dict(                # dropped -> (kept, r)
            correlated, orient="index", columns=["kept", "abs_r"]
        ),
    }

    return drop_cols, report

# Clean Dataset

some compositions are not compatible with the featurisation method

In [3]:
df = pd.read_csv("adj_exp_dataset.csv")
df

,UID,Chemical,Time,Temperature,P2O3ratio,Cat_Success,Impurity_Percentage
0,A00006,NaCu0.275073Mn0.724927O2,12,950,1.000000,Successful,43.56
1,A00008,NaFe0.497381Mn0.502619O2,12,950,0.997606,Successful,63.63
2,A00010,NaMn0.736534Ni0.263466O2,12,950,0.978199,Successful,31.24
3,A00011,NaCu0.250091Fe0.523806Mn0.226103O2,12,950,0.627143,Successful,41.06
4,A00013,NaCu0.3296Mn0.63802Ni0.03238O2,12,950,0.980703,Successful,56.44
...,...,...,...,...,...,...,...
552,U00365,Na0.5Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,3,950,-1.000000,Unsuccessful,88.61
553,U00366,Na0.5Fe0.388889Mn0.388889Cu0.111111Ni0.111111O2,3,950,0.263612,Successful,16.42
554,U00367,Na0.5Cu0.388889Fe0.388889Mn0.111111Ni0.111111O2,3,950,0.140791,Successful,49.99
555,U00368,Na0.5Mn0.388889Ni0.388889Fe0.111111Cu0.111111O2,3,950,0.730366,Successful,44.87


In [4]:
for formula in df["Chemical"]:
    try:
        if charge_balance(formula) is None:
            print(f"Failed: {formula} | unknown element")
    except Exception as e:
        print(f"Failed: {formula} | [{type(e).__name__}]: {e}")

In [5]:
def charge_balance_ok(formula):
    try:
        return charge_balance(formula) is not None
    except Exception:
        return False

mask = df["Chemical"].apply(charge_balance_ok)
print(f"Kept {mask.sum()} / {len(mask)} formulas")
df = df[mask].reset_index(drop=True)
df

Kept 557 / 557 formulas


,UID,Chemical,Time,Temperature,P2O3ratio,Cat_Success,Impurity_Percentage
0,A00006,NaCu0.275073Mn0.724927O2,12,950,1.000000,Successful,43.56
1,A00008,NaFe0.497381Mn0.502619O2,12,950,0.997606,Successful,63.63
2,A00010,NaMn0.736534Ni0.263466O2,12,950,0.978199,Successful,31.24
3,A00011,NaCu0.250091Fe0.523806Mn0.226103O2,12,950,0.627143,Successful,41.06
4,A00013,NaCu0.3296Mn0.63802Ni0.03238O2,12,950,0.980703,Successful,56.44
...,...,...,...,...,...,...,...
552,U00365,Na0.5Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,3,950,-1.000000,Unsuccessful,88.61
553,U00366,Na0.5Fe0.388889Mn0.388889Cu0.111111Ni0.111111O2,3,950,0.263612,Successful,16.42
554,U00367,Na0.5Cu0.388889Fe0.388889Mn0.111111Ni0.111111O2,3,950,0.140791,Successful,49.99
555,U00368,Na0.5Mn0.388889Ni0.388889Fe0.111111Cu0.111111O2,3,950,0.730366,Successful,44.87


# Featurise the dataset using approximate

In [6]:
def featurize_df(df, extractor, formula_col="Compound"):
    formulas = df[formula_col].tolist()
    feature_rows = extractor.featurize_many(formulas)
    feat_df = pd.DataFrame(feature_rows, index=df.index)

    # Keep only rows that featurized successfully
    feature_cols = feat_df.drop(columns=["formula"], errors="ignore")
    if feature_cols.shape[1] == 0:
        return df.iloc[0:0].copy()

    valid_rows = feature_cols.notna().any(axis=1)
    df_valid = df.loc[valid_rows]
    feat_valid = feat_df.loc[valid_rows]

    failed_formulas = df.loc[~valid_rows, formula_col]
    for formula in failed_formulas:
        print("Formula Fail:", formula)

    feature_names = [c for c in feat_df.columns if c != "formula"]

    return df_valid.join(feat_valid), feature_names

# Initialise the featuriser
extractor = MaterialFeatureExtractor(
    mode="both",
    ionic_radius_unit="pm",
    enable_tm_potential=True,
    tm_cation="Na",
    tm_anion="O",
)

df_feat, feat_cols = featurize_df(df, extractor, formula_col="Chemical")
df_feat

,UID,Chemical,Time,Temperature,P2O3ratio,Cat_Success,Impurity_Percentage,formula,all_valence_s_sum,all_valence_s_avg,...,var_gordy_en_max,var_gordy_en_range,var_gordy_en_mode,var_mb_en_sum,var_mb_en_avg,var_mb_en_dev,var_mb_en_min,var_mb_en_max,var_mb_en_range,var_mb_en_mode
0,A00006,NaCu0.275073Mn0.724927O2,12,950,1.000000,Successful,43.56,NaCu0.275073Mn0.724927O2,4.0,1.000000,...,0.298507,0.043730,0.254777,0.063692,0.063692,0.024618,0.044776,0.103448,0.058672,0.050955
1,A00008,NaFe0.497381Mn0.502619O2,12,950,0.997606,Successful,63.63,NaFe0.497381Mn0.502619O2,4.0,1.000000,...,0.266255,0.011477,0.254777,0.057287,0.057287,0.006369,0.050955,0.063694,0.012739,0.050955
2,A00010,NaMn0.736534Ni0.263466O2,12,950,0.978199,Successful,31.24,NaMn0.736534Ni0.263466O2,4.0,1.000000,...,0.298507,0.043730,0.254777,0.061272,0.061272,0.021129,0.044776,0.096386,0.051609,0.050955
3,A00011,NaCu0.250091Fe0.523806Mn0.226103O2,12,950,0.627143,Successful,41.06,NaCu0.250091Fe0.523806Mn0.226103O2,4.0,1.000000,...,0.333333,0.067079,0.266255,0.068999,0.068999,0.021850,0.000000,0.103448,0.103448,0.063694
4,A00013,NaCu0.3296Mn0.63802Ni0.03238O2,12,950,0.980703,Successful,56.44,NaCu0.3296Mn0.63802Ni0.03238O2,4.0,1.000000,...,0.298507,0.043730,0.298507,0.067495,0.067495,0.026751,0.044776,0.103448,0.058672,0.044776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552,U00365,Na0.5Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,3,950,-1.000000,Unsuccessful,88.61,Na0.5Cu0.388889Ni0.388889Fe0.111111Mn0.111111O2,4.0,1.142857,...,0.366266,0.100012,0.271435,0.084957,0.084957,0.032229,0.000000,0.103448,0.103448,0.103448
553,U00366,Na0.5Fe0.388889Mn0.388889Cu0.111111Ni0.111111O2,3,950,0.263612,Successful,16.42,Na0.5Fe0.388889Mn0.388889Cu0.111111Ni0.111111O2,4.0,1.142857,...,0.366266,0.100012,0.266255,0.062763,0.062763,0.023919,0.000000,0.103448,0.103448,0.063694
554,U00367,Na0.5Cu0.388889Fe0.388889Mn0.111111Ni0.111111O2,3,950,0.140791,Successful,49.99,Na0.5Cu0.388889Fe0.388889Mn0.111111Ni0.111111O2,4.0,1.142857,...,0.366266,0.100012,0.271435,0.075760,0.075760,0.032264,0.000000,0.103448,0.103448,0.103448
555,U00368,Na0.5Mn0.388889Ni0.388889Fe0.111111Cu0.111111O2,3,950,0.730366,Successful,44.87,Na0.5Mn0.388889Ni0.388889Fe0.111111Cu0.111111O2,4.0,1.142857,...,0.366266,0.100012,0.298507,0.073252,0.073252,0.025014,0.044776,0.103448,0.058672,0.044776


In [7]:
df_feat, feat_cols = featurize_df(df, extractor, formula_col="Chemical")
drop_cols, report = clean_features(df_feat, feat_cols)
feat_cols_clean = [c for c in feat_cols if c not in drop_cols]

report["correlated"]

,kept,abs_r
all_unfilled_valence_s_sum,all_valence_s_avg,0.9992
all_unfilled_valence_s_avg,all_valence_s_avg,1.0000
all_unfilled_valence_s_dev,all_valence_s_dev,1.0000
all_unfilled_valence_s_mode,all_valence_s_mode,1.0000
all_valence_p_avg,all_valence_s_avg,1.0000
...,...,...
var_pauling_en_avg,var_pauling_en_sum,0.9999
var_gordy_en_avg,var_gordy_en_sum,0.9999
var_mb_en_sum,all_mb_en_sum,1.0000
var_mb_en_avg,all_mb_en_sum,1.0000


In [8]:
def add_parent_uid(df, uid_col="UID", parent_col="parent_uid"):
    """Add a column linking every row to its U00 parent.
 
    A00xxxx -> U00xxxx (same number); U00 rows point to themselves.
    Used later to keep each A00 row in the same fold as its parent.
    """
    df = df.copy()
    uids = df[uid_col].astype(str)
    df[parent_col] = np.where(uids.str.startswith("A00"), "U" + uids.str[1:], uids)
    return df
 
def check_orphans(df, uid_col="UID", parent_col="parent_uid"):
    """A00 rows whose U00 parent is missing (e.g. dropped by charge balance)."""
    uids = df[uid_col].astype(str)
    u00_ids = set(uids[uids.str.startswith("U00")])
    is_a00 = uids.str.startswith("A00")
    orphans = df[is_a00 & ~df[parent_col].isin(u00_ids)]
    print(f"{len(orphans)} orphaned A00 rows")
    return orphans

df_feat = add_parent_uid(df_feat)
orphans = check_orphans(df_feat)
orphans[["UID", "parent_uid", "Chemical"]]

0 orphaned A00 rows


,UID,parent_uid,Chemical
